# Is the bottom-sticking caused by the W fix?

Hypothesis (supervisor): the vertical velocity `W` was recomputed to remove the
free-surface (eta) breathing and redistribute it down the column (`Fix_W.ipynb`).
If that introduced a downward bias, particles would be pushed onto - or through -
the seafloor, which is what the E groups do (~42% of all particles ground on the
bottom).

The `Fix_W` recipe, for reference (z positive up, H>0, eta positive up):

$$W_{merc}(z)=W_\eta(z)+W_{fixed}(z),\qquad
  W_\eta(z)=\frac{z+H}{H}\,\frac{\partial\eta}{\partial t},\qquad
  W_{fixed}=W_{merc}-W_\eta$$

`deta_dt` is taken as the model's surface `W`. Note the weight `(z+H)/H` is **1
at the surface and 0 at the bottom** - so by construction the correction is
largest at the surface and vanishes at the seafloor. That already predicts the
fix cannot change the near-bottom velocity. This notebook checks it against the
data and separates three questions:

1. Did the fix introduce a net vertical bias?  (compare fixed vs original W)
2. Is there a downward bias on the shallow shelf where particles stick?
3. Do particles sink gradually (W-driven) or fall *below* the bathymetry?


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
from joblib import Parallel, delayed
import config as C

# --- TUNE ME -----------------------------------------------------------------
MONTH      = "2005-06"      # a month to inspect the W field
SHELF_BOX  = dict(lon=(-53, -49), lat=(5, 8), max_levels=15)  # Guiana shelf
N_DESCENT  = 200            # E0 particles for the descent / below-bottom test
# -----------------------------------------------------------------------------

UVW   = "/work/bk1450/b383184/Amazon/Mercator/data/variables_c/UVW"
W_RAW = "/work/bk1450/b383184/Amazon/Mercator/data/variables"
MESH_Z = "/work/bk1450/b383184/Amazon/Mercator/data/Zgr_cmesh2.nc"
MESH_H = "/work/bk1450/b383184/Amazon/Mercator/data/Hgr_cmesh.nc"
N_IO   = int(os.environ.get("SLURM_CPUS_PER_TASK", 16))
inv = {v: k for k, v in C.GROUP_NAMES.items()}
store_paths = {p.stem: p for p in C.list_stores()}
print("month:", MONTH, "| shelf box:", SHELF_BOX)

## Model grid and bathymetry
`mbathy` (wet levels) + `gdepw_0` give the seafloor depth; used both to locate
the shelf and to test whether particles go below the bottom.

In [ ]:
Zg = xr.open_dataset(MESH_Z); Hg = xr.open_dataset(MESH_H)
mbathy = np.asarray(Zg["mbathy"]).squeeze().astype(int)
gdepw  = np.asarray(Zg["gdepw_0"]).squeeze()
glamt  = np.asarray(Hg["glamt"]).squeeze(); gphit = np.asarray(Hg["gphit"]).squeeze()
bottom = np.where(mbathy > 0, gdepw[np.clip(mbathy, 0, len(gdepw) - 1)], np.nan)
grid_tree = cKDTree(np.c_[glamt.ravel(), gphit.ravel()])
def bottom_at(lat, lon):
    _, i = grid_tree.query(np.c_[np.atleast_1d(lon), np.atleast_1d(lat)])
    return bottom.ravel()[i]
print("grid", glamt.shape, "| seafloor range",
      f"{np.nanmin(bottom):.1f} .. {np.nanmax(bottom):.0f} m")

## 1. Did the fix introduce a net vertical bias?
Compare the fixed W (`W_*fc.nc`) with the original (`variables/W_*.nc`), per
depth level. If the fix biased the column downward, the fixed mean would be
systematically more negative (down).

In [ ]:
wf = xr.open_dataset(f"{UVW}/W_{MONTH}fc.nc")["vovecrtz"].values
wo = xr.open_dataset(f"{W_RAW}/W_{MONTH}.nc")["vovecrtz"].values
print(f"original W: mean {np.nanmean(wo):+.2e} m/s")
print(f"fixed    W: mean {np.nanmean(wf):+.2e} m/s")
print(f"net shift from the fix: {np.nanmean(wf)-np.nanmean(wo):+.2e} m/s "
      f"({(np.nanmean(wf)-np.nanmean(wo))*86400:+.4f} m/day)  <- ~0 means unbiased")

nlev = wo.shape[1]
mo = [np.nanmean(wo[:, k]) for k in range(nlev)]
mf = [np.nanmean(wf[:, k]) for k in range(nlev)]
depth = gdepw[:nlev]
fig, ax = plt.subplots(figsize=(6, 7))
ax.plot(np.array(mo)*86400, depth, "-o", ms=3, label="original W", color="#2a78d6")
ax.plot(np.array(mf)*86400, depth, "-o", ms=3, label="fixed W", color="#eb6834")
ax.axvline(0, color="0.6", lw=.8)
ax.invert_yaxis(); ax.set_xlabel("horizontal-mean W (m/day)"); ax.set_ylabel("depth (m)")
ax.legend(frameon=False); ax.set_title("Mean W by depth: fix is surface-concentrated",
                                       loc="left", fontsize=11)
for s in ("top", "right"): ax.spines[s].set_visible(False)
ax.grid(color="0.92"); ax.set_axisbelow(True)
fig.tight_layout(); plt.show()

print(f"\ncorrection at surface (lev 0): {(mf[0]-mo[0])*86400:+.3f} m/day")
print(f"correction near bottom (lev 20): {(mf[20]-mo[20])*86400:+.3f} m/day")

## 2. Downward bias on the shallow shelf?
The domain mean can hide a shelf-specific problem. Take the shallow Guiana-shelf
columns where the E particles pile up and read the **time-mean W in each
column's bottom cell**. A spurious sink would show a persistent negative
(downward) value.

In [ ]:
shelf = ((glamt > SHELF_BOX["lon"][0]) & (glamt < SHELF_BOX["lon"][1])
         & (gphit > SHELF_BOX["lat"][0]) & (gphit < SHELF_BOX["lat"][1])
         & (mbathy > 0) & (mbathy < SHELF_BOX["max_levels"]))
wf_tmean = wf.mean(axis=0)                       # (depth, y, x)
jj, ii = np.where(shelf)
wbot = np.array([wf_tmean[min(mbathy[j, i] - 1, wf.shape[1] - 1), j, i]
                 for j, i in zip(jj, ii)])
wbot = wbot[np.isfinite(wbot)]
print(f"shallow shelf columns in box: {shelf.sum()}")
print(f"time-mean W in the bottom cell: {wbot.mean()*86400:+.3f} m/day "
      f"(median {np.median(wbot)*86400:+.3f})")
print(f"  downward (W<0) fraction: {100*np.mean(wbot < 0):.0f}%")
print("  -> a persistent sink would be strongly negative; near-zero/positive rules it out")

## 3. Do particles sink gradually, or fall below the seafloor?
Stream E0 paths. `descent` = does depth increase smoothly toward the bottom
(consistent with weak vertical advection), and does any particle end up more
than a couple of metres **below** the model seafloor (which would point at a
bottom-boundary problem rather than physics)?

In [ ]:
s = lab_E0 = None
lab = pd.read_parquet(C.LABELED_FILE,
                      columns=["trajectory_id", "store", "cluster_group", "status"])
lab = lab[(lab.status == "complete") & (lab.cluster_group >= 0)]
s = lab[lab.cluster_group == inv["E0"]].sample(N_DESCENT, random_state=3).copy()
s["local"] = s.trajectory_id % C.TRAJ_PER_STORE

def _descent(name, grp):
    ds = xr.open_zarr(store_paths[name]); out = []
    for l, tid in zip(grp.local.to_numpy(), grp.trajectory_id.to_numpy()):
        lo, la, z = ds.lon.values[l], ds.lat.values[l], ds.z.values[l]
        t = ds.time.values[l]; m = ~np.isnan(lo) & ~np.isnat(t)
        lo, la, z = lo[m], la[m], z[m]
        H = bottom_at(la, lo)
        out.append(dict(tid=int(tid), z0=z[0], zmax=float(np.nanmax(z)),
                        below=float(np.nanmax(z - H)),
                        day=(t[m] - t[m][0]) / np.timedelta64(1, "D"),
                        z=z, Hend=float(H[-1])))
    return out

desc = [x for part in Parallel(n_jobs=N_IO, backend="threading")(
            delayed(_descent)(n, g) for n, g in s.groupby("store")) for x in part]
below = np.array([d["below"] for d in desc])
print(f"E0: start depth med {np.median([d['z0'] for d in desc]):.1f} m "
      f"-> max depth med {np.median([d['zmax'] for d in desc]):.1f} m")
print(f"particles ever >2 m BELOW the seafloor: {int((below > 2).sum())}/{len(desc)} "
      f"= {100*(below > 2).mean():.0f}%")
print(f"amount below seafloor: median {np.median(below):+.1f}  "
      f"p90 {np.percentile(below, 90):+.1f}  max {below.max():+.1f} m")

fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
for d in desc[:40]:
    ax[0].plot(d["day"], d["z"], color="#eb6834", lw=.6, alpha=.5)
ax[0].invert_yaxis(); ax[0].set_xlabel("days since release"); ax[0].set_ylabel("depth (m)")
ax[0].set_title("E0 depth vs time (40 particles)", loc="left", fontsize=10)
ax[1].hist(below, bins=40, color="#2a78d6")
ax[1].axvline(0, color="k", lw=1)
ax[1].set_xlabel("max depth below the model seafloor (m)")
ax[1].set_ylabel("particles")
ax[1].set_title("penetration below bathymetry", loc="left", fontsize=10)
for a in ax:
    for sp in ("top", "right"): a.spines[sp].set_visible(False)
    a.grid(color="0.92"); a.set_axisbelow(True)
fig.tight_layout(); plt.show()

## Verdict

Read the three results above. Expected pattern if **Fix_W is innocent**:

1. net W shift ~ 0 and the correction concentrated at the surface;
2. shelf bottom-cell W near zero / not systematically downward;
3. particles descend slowly and mostly sit *at* the seafloor, with only a small
   fraction dipping below it.

If instead the fix were biased you would see a clearly negative net shift and a
strong downward W on the shelf. The small sub-seafloor penetration that remains
is a **Parcels bottom-boundary effect** - `AdvectionRK4_3D` has no reflective
bottom, so a particle in the deepest wet cell can be carried a little below the
last w-level - and is unrelated to the W recomputation. It is the same
no-beaching-kernel limitation already documented, not a mistake in `Fix_W`.
